# BDA700 — Week 8
## Streamlit App Development I: From Model to Interactive App

**This notebook is designed to run in Google Colab.** Upload it via **File → Upload notebook** at [colab.research.google.com](https://colab.research.google.com). Self-contained.

---

### Learning objectives
- Recap the ML lifecycle and serialize a model: `joblib`
- Understand Streamlit's script-rerun execution model and why caching matters
- Build a Streamlit app that loads a serialized pipeline and serves live predictions
- Use `st.session_state` to persist data across reruns
- Deploy a Streamlit app for real, with a permanent URL, on Streamlit Community Cloud




## 0. Environment Setup

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("environment ready")

environment ready


In [7]:
import sys
import joblib
import seaborn as sns
import pandas as pd
import numpy as np
import streamlit as st
import sklearn

print("Python version:", sys.version)
print("numpy version: ", np.__version__)
print("pandas version:", pd.__version__)
print("seaborn version:", sns.__version__)
print("Joblib version:", joblib.__version__)
print("Streamlit version:", st.__version__)

Python version: 3.10.0 (v3.10.0:b494f5935c, Oct  4 2021, 14:59:19) [Clang 12.0.5 (clang-1205.0.22.11)]
numpy version:  1.26.4
pandas version: 2.2.3
seaborn version: 0.13.2
Joblib version: 1.5.0
Streamlit version: 1.65.0


In [5]:
print(sys.executable)

/usr/local/bin/python3.10


## 2. Model Serialization: `joblib`

In [12]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
class_names = iris.target_names

# TRAINING
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000)),
])
model_pipeline.fit(X_train, y_train)

# VALIDATION
val_accuracy = accuracy_score(y_test, model_pipeline.predict(X_test))
print(f"validation accuracy: {val_accuracy:.3f}")

STAGING_THRESHOLD = 0.90
if val_accuracy >= STAGING_THRESHOLD:
    print(f"✅ meets staging threshold ({STAGING_THRESHOLD}) — promoting to staging")
else:
    print(f"❌ below staging threshold ({STAGING_THRESHOLD}) — do NOT promote")

validation accuracy: 0.933
✅ meets staging threshold (0.9) — promoting to staging


In [13]:
import joblib

joblib.dump(model_pipeline, "iris_pipeline.joblib")
print("saved iris_pipeline.joblib")

saved iris_pipeline.joblib


## 3. Streamlit App Architecture

Streamlit is not a web framework in the traditional sense — there is no router, no request handler you write yourself. Instead, **Streamlit reruns your entire script, top to bottom, every time the user interacts with a widget** (moves a slider, clicks a button). That mental model explains almost everything about how Streamlit code is written:

- Anything expensive (loading a model, reading a large file) needs to survive reruns — that's what **caching** is for.
- Anything you want to persist *across* reruns that isn't naturally re-derived (like a running log of predictions) needs `st.session_state`.
- Layout is just Python control flow — an `if` block that doesn't run simply doesn't render that part of the page.

Two caching decorators cover almost every case:

| Decorator | Use for | Example |
|---|---|---|
| `@st.cache_resource` | Objects that shouldn't be re-created — unserializable, expensive-to-build | ML models, DB connections, API clients |
| `@st.cache_data` | Data that can be serialized and compared by value | DataFrames, arrays, results of `pd.read_csv` |

Using the wrong one is a common bug: `@st.cache_data` on a scikit-learn pipeline will try to hash/serialize it on every call (slow, sometimes fails outright); `@st.cache_resource` is what you want for the model itself.

In [14]:
%%writefile app.py
import joblib
import numpy as np
import streamlit as st

st.set_page_config(page_title="Iris Predictor", page_icon="🌸", layout="centered")


@st.cache_resource
def load_model():
    return joblib.load("iris_pipeline.joblib")


model = load_model()
CLASS_NAMES = ["setosa", "versicolor", "virginica"]

st.title("🌸 Iris Species Predictor")
st.caption("Loads iris_pipeline.joblib directly — no API yet.")

if "history" not in st.session_state:
    st.session_state.history = []

#syntax st.slider("title", min, max, default)
sepal_length = st.slider("Sepal length (cm)", 4.0, 8.0, 5.1)
sepal_width = st.slider("Sepal width (cm)", 2.0, 4.5, 3.5)
petal_length = st.slider("Petal length (cm)", 1.0, 7.0, 1.4)
petal_width = st.slider("Petal width (cm)", 0.1, 2.5, 0.2)

if st.button("Predict species"):
    row = np.array([[sepal_length, sepal_width, petal_length, petal_width]])
    pred_class = int(model.predict(row)[0])
    pred_proba = model.predict_proba(row)[0]

    st.success(f"Predicted species: **{CLASS_NAMES[pred_class]}**")
    st.bar_chart(dict(zip(CLASS_NAMES, pred_proba)))

    st.session_state.history.append(CLASS_NAMES[pred_class])

if st.session_state.history:
    st.caption(f"Predictions this session: {', '.join(st.session_state.history)}")

Overwriting app.py


Notice what `st.session_state.history` buys you: without it, `st.bar_chart` and the success message would still work fine (they redraw every rerun regardless), but the running list of "predictions this session" would reset to empty on every single slider movement, because reruns re-execute the whole script from scratch. `st.session_state` is the one thing that survives a rerun by design.

### 4. Run the app live



In [16]:
# 1. Terminate any running Streamlit processes
!pkill -f streamlit 2>/dev/null

# 2. Start Streamlit in the background
import subprocess, time
time.sleep(1)
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
])
time.sleep(5)



  Stopping...


2026-11-05 17:09:04.489 Uvicorn server started on :::8501



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://10.248.57.229:8501
  External URL: http://142.204.17.59:8501

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            


> **Troubleshooting:** if you see `TypeError: Failed to fetch dynamically imported module: ...` (rare with Colab's own port forwarding, but possible), your browser tab is holding assets from a previous run. Re-run the cell (it kills any leftover process first) and click the freshly generated link — don't reuse an old tab.
>
> **Note:** `serve_kernel_port_as_window` only works when this notebook is actually running on Colab's hosted runtime (not a local runtime). If you're running locally instead, use `streamlit run app.py` in a terminal and open `http://localhost:8501` directly.



### 🔹 Mini-exercise 1

Add a fifth widget to `app.py`: a `st.selectbox` letting the user pick between "Standard" and "Detailed" result view. In "Detailed" mode, also show the raw probability numbers with `st.write` (not just the bar chart). Rewrite the cell above with `%%writefile app.py` again to update the file, then move on — you'll see it live in the optional run further down.

In [ ]:
# TODO: add the selectbox and detailed view to app.py


## 5. Deploying for Real: Streamlit Community Cloud

Your Course Project needs a permanent URL, and that means **Streamlit Community Cloud** (share.streamlit.io): a free, always-on host built specifically for Streamlit apps.

**The path from repo to live URL:**

1. **Structure your project as a GitHub repo.** At minimum: `app.py`, a `requirements.txt` with pinned versions, and either your serialized model file committed to the repo (fine for a small `.joblib` like ours) or the code to rebuild it on startup.
2. **Push to GitHub.** Public repos are free and simplest; Community Cloud can also deploy from a private repo once you connect your GitHub account.
3. **Go to [share.streamlit.io](https://share.streamlit.io) → "New app".** Pick your repo, branch, and the path to `app.py`. Deploy.
4. **The build reads `requirements.txt` and installs it in a clean container** — this is exactly why we've pinned every version all semester: an unpinned `scikit-learn` could resolve to a version your `.joblib` file wasn't trained under, and loading would fail or silently mispredict.
5. **Secrets** (API keys, database URLs — not needed for this simple app, but common in real projects) go in the app's **Settings → Secrets** in the Community Cloud dashboard, formatted as TOML, and are read in-app via `st.secrets["your_key"]`. Never commit secrets to the repo itself.

**About the FastAPI piece, specifically:** Community Cloud hosts your Streamlit app — it does not host an arbitrary second FastAPI process alongside it. Today's API exercise teaches the REST inference pattern, which is genuinely how many production systems separate serving from the front end. But for **your Course Project's deployed app**, the simplest and recommended architecture is what you built *last week*: the Streamlit app loads the serialized pipeline directly with `joblib.load` (wrapped in `@st.cache_resource`), no separate API required. Only reach for a separately-hosted FastAPI backend if your project genuinely needs the same model served to something other than the Streamlit app (e.g. a mobile client) — and if so, that backend needs its own always-on host (a small cloud VM or a managed API platform), since Community Cloud won't run it for you.